In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# Milestone 3 setup (run first, before answering any questions)
# Run this code in a code cell
#  before answering any of the question. This will create your knowledge 
# base and the FAISS index for this milestone.

!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 88.1 MB/s eta 0:00:00
Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


**QUESTION_1:Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)**

In [3]:
from transformers import pipeline

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row = train.iloc[150]

prompt = str(row["prompt"])

labels = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

result = zs(prompt, candidate_labels=labels)

correct_text = str(row[row["answer"]])

idx = result["labels"].index(correct_text)

print("Correct Option:", row["answer"])
print("Probability:", round(result["scores"][idx], 3))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Correct Option: C
Probability: 0.384


**QUESTION_2:Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?**

In [4]:
import numpy as np

# Query is the prompt at row 150
query = train.iloc[150]["prompt"]

# Embed the query
query_embedding = model.encode([query])

# Search the FAISS index
D, I = index.search(query_embedding, 10)

print("Top-10 Retrieved Document Indices:")
print(I[0])

# Find the true document (the correct answer text of row 150)
true_doc = kb[150]

rank = None

for i, idx in enumerate(I[0]):
    if kb[idx] == true_doc:
        rank = i + 1
        break

print("Rank =", rank)

Top-10 Retrieved Document Indices:
[ 663 1701 1269 1532  576  847 1693 1906  168  150]
Rank = 10


**QUESTION_3: Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?**

In [5]:
import numpy as np
from sentence_transformers import CrossEncoder

# Prompt from row 150
prompt_150 = train.iloc[150]["prompt"]

# Embed the prompt
query_embedding = model.encode([prompt_150])

# Retrieve top 10 using FAISS
distances, retrieved_indices = index.search(query_embedding, 10)

print("Retrieved indices:")
print(retrieved_indices[0])

# Get retrieved documents
docs_10 = [kb[i] for i in retrieved_indices[0]]

# Load Cross Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Create prompt-document pairs
pairs = [[prompt_150, doc] for doc in docs_10]

# Predict relevance scores
scores = cross_encoder.predict(pairs)

# Sort by CrossEncoder score (highest first)
ranking = sorted(
    zip(retrieved_indices[0], docs_10, scores),
    key=lambda x: x[2],
    reverse=True
)

print("\nCross Encoder Ranking\n")

for rank, (idx, doc, score) in enumerate(ranking, start=1):
    print(rank, idx, round(float(score),4))

# Find rank of true document
true_doc = kb[150]

for rank, (_, doc, _) in enumerate(ranking, start=1):
    if doc == true_doc:
        print("\nFinal Rank =", rank)
        break

Retrieved indices:
[ 663 1701 1269 1532  576  847 1693 1906  168  150]


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


Cross Encoder Ranking

1 150 4.7585
2 847 4.7526
3 1693 4.7526
4 1906 4.7526
5 1269 4.7375
6 1532 4.7375
7 168 4.7072
8 576 4.687
9 663 4.6602
10 1701 4.6602

Final Rank = 1


**QUESTION_4:Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?**

In [6]:
from transformers import AutoTokenizer

# BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Query prompt
prompt = train.iloc[42]["prompt"]

# Retrieve top-5 documents
query_embedding = model.encode([prompt])

D, I = index.search(query_embedding, 5)

docs = [kb[i] for i in I[0]]

# Concatenate with one space
context = " ".join(docs)

# Final string
text = f"Context: {context} Question: {prompt}"

# Tokenize WITHOUT truncation
tokens = tokenizer(text)

print("Number of tokens =", len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Number of tokens = 216


**QUESTION_5:Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).**

In [7]:
from transformers import pipeline

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row = train.iloc[150]

prompt = row["prompt"]

true_doc = kb[150]

rag_prompt = f"Context: {true_doc} Question: {prompt}"

labels = [
    row["A"],
    row["B"],
    row["C"],
    row["D"],
    row["E"]
]

result = zs(
    rag_prompt,
    candidate_labels=labels
)

correct_text = row[row["answer"]]

idx = result["labels"].index(correct_text)

print(result)

print("Probability =", round(result["scores"][idx],3))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on 

**QUESTION_6: What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).**

In [8]:
from transformers import pipeline

# Zero-shot classifier
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row = train.iloc[150]

prompt = str(row["prompt"])

# Wrong context
wrong_doc = kb[999]

rag_prompt = f"Context: {wrong_doc} Question: {prompt}"

labels = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

answer = row["answer"]

result = zs(
    rag_prompt,
    candidate_labels=labels
)

correct_option = str(row[answer])

idx = result["labels"].index(correct_option)

print(result)

print("\nCorrect Answer:", answer)
print("Probability =", round(result["scores"][idx],3))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'Context: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics. Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz i

**QUESTION_7:For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).**

In [9]:
hits = 0

for i in range(100):

    prompt = train.iloc[i]["prompt"]

    query_embedding = model.encode([prompt])

    D, I = index.search(query_embedding, 5)

    retrieved_docs = [kb[idx] for idx in I[0]]

    true_doc = kb[i]

    if true_doc in retrieved_docs:
        hits += 1

hit_rate = hits / 100 * 100

print("Hits =", hits)
print("Hit Rate =", round(hit_rate,1))

Hits = 73
Hit Rate = 73.0


**QUESTION_8: Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score:
 Look at the probability scores output by the model. Rank the options 
from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).**

In [10]:
from sentence_transformers import CrossEncoder
from transformers import pipeline
import numpy as np

# Load models
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

map3_scores = []

for i in range(20):

    row = train.iloc[i]

    prompt = str(row["prompt"])

    
    # Retrieve Top-5 using FAISS
   

    query_embedding = model.encode([prompt])

    D, I = index.search(query_embedding, 5)

    docs = [kb[idx] for idx in I[0]]

   
    # Re-rank with CrossEncoder
  

    pairs = [[prompt, doc] for doc in docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]

    # RAG Prompt
  

    rag_prompt = f"Context: {best_doc} Question: {prompt}"

   
    # Zero-shot Prediction
    

    option_letters = ["A","B","C","D","E"]

    option_texts = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    result = zs(
        rag_prompt,
        candidate_labels=option_texts
    )

   
    # Convert predicted texts back to option letters
   

    ranked_letters = []

    for label in result["labels"]:

        for letter, text in zip(option_letters, option_texts):

            if label == text:
                ranked_letters.append(letter)
                break

    top3 = ranked_letters[:3]

    actual = row["answer"]

    if actual in top3:

        rank = top3.index(actual) + 1

        score = 1 / rank

    else:

        score = 0

    map3_scores.append(score)

    print(
        f"Row {i:02d}",
        "Actual:", actual,
        "Top3:", top3,
        "MAP3:", score
    )


# Final Answer


final_map3 = np.mean(map3_scores)


print("FINAL MAP@3 =", round(final_map3,3))


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Row 00 Actual: B Top3: ['B', 'D', 'A'] MAP3: 1.0
Row 01 Actual: A Top3: ['A', 'E', 'C'] MAP3: 1.0
Row 02 Actual: C Top3: ['C', 'D', 'B'] MAP3: 1.0
Row 03 Actual: B Top3: ['B', 'D', 'A'] MAP3: 1.0
Row 04 Actual: A Top3: ['A', 'B', 'C'] MAP3: 1.0
Row 05 Actual: C Top3: ['B', 'C', 'A'] MAP3: 0.5
Row 06 Actual: E Top3: ['E', 'B', 'D'] MAP3: 1.0
Row 07 Actual: A Top3: ['A', 'B', 'C'] MAP3: 1.0
Row 08 Actual: A Top3: ['A', 'C', 'D'] MAP3: 1.0


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Row 09 Actual: A Top3: ['A', 'B', 'C'] MAP3: 1.0
Row 10 Actual: C Top3: ['C', 'A', 'D'] MAP3: 1.0
Row 11 Actual: B Top3: ['B', 'C', 'A'] MAP3: 1.0
Row 12 Actual: D Top3: ['D', 'A', 'B'] MAP3: 1.0
Row 13 Actual: E Top3: ['E', 'D', 'B'] MAP3: 1.0
Row 14 Actual: E Top3: ['E', 'A', 'D'] MAP3: 1.0
Row 15 Actual: E Top3: ['E', 'B', 'C'] MAP3: 1.0
Row 16 Actual: C Top3: ['C', 'B', 'E'] MAP3: 1.0
Row 17 Actual: C Top3: ['C', 'B', 'E'] MAP3: 1.0
Row 18 Actual: B Top3: ['B', 'D', 'A'] MAP3: 1.0
Row 19 Actual: C Top3: ['C', 'D', 'A'] MAP3: 1.0
FINAL MAP@3 = 0.975
